## **BARBADOS TRAFFIC SOLUTION**

### *By Team Traffic Solvers (NYMFREE & KEYSTATS)*

The goal of this competition was to uncover the **root causes of traffic congestion** at the NormanNile Roundabout and propose insights that can support **informed, data-driven decision making**.

Our approach went beyond surface-level analysis. We explored the problem **from multiple angles**, capturing both **natural, unchangeable factors** (such as road structure and time-based patterns) and **human-related causes** like reckless driving behavior and peak-hour congestion. We also examined **historical traffic trends** alongside **coincidental or event-driven scenarios**—for example, specific times or occasions when traffic volume spikes because more people tend to use that route. In short, our solution was designed to view the traffic problem as a **dynamic system**, influenced by many interacting factors rather than a single cause.

To keep the work **clear, structured, and easy to follow**—and to avoid an extremely long and overwhelming notebook—we divided our solution into **six separate notebooks**. Each notebook represents a distinct stage of our thinking process and builds logically toward the next. Together, they form a complete, step-by-step narrative of how our solution was developed.

### **Important Note (NB):**

To ensure smooth execution of the code **without repeatedly adjusting file paths**, we have packaged all notebooks and required CSV files into a **single ZIP file**. After extracting the files locally, you will find the **seven notebooks clearly numbered**, indicating the correct order in which they should be run.

The recommended environment for running the notebooks is **Lightning AI with a T4 GPU**.

To get started, you only need to upload the following files:

* `Train.csv`
* `TestInputSegments.csv`
* `SampleSubmission.csv`

Once these are in place, the entire pipeline can be executed seamlessly.


### Installing Essential Libraries

In [1]:
!pip install ultralytics supervision 
!pip install optuna
!pip install scikit-learn
!pip install lightgbm xgboost
!pip install pandas


INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 51.2 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 57.2 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 102.4 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 9.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 24.2 MB/s  0:00:14m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 27.2 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 28.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━

### Loading Train original Data

In [2]:
import pandas as pd
Train=pd.read_csv("Train.csv")
display(Train.head())

,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train


In [3]:
Train['datetimestamp_start'] = pd.to_datetime(Train['datetimestamp_start'])
Train['datetimestamp_end'] = pd.to_datetime(Train['datetimestamp_end'])

Train.info()

<class 'pandas.DataFrame'>
RangeIndex: 16076 entries, 0 to 16075
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   responseId               16076 non-null  str           
 1   view_label               16076 non-null  str           
 2   ID_enter                 16076 non-null  str           
 3   ID_exit                  16076 non-null  str           
 4   videos                   16076 non-null  str           
 5   video_time               16076 non-null  str           
 6   datetimestamp_start      16076 non-null  datetime64[us]
 7   datetimestamp_end        16076 non-null  datetime64[us]
 8   date                     16076 non-null  str           
 9   signaling                16076 non-null  str           
 10  congestion_enter_rating  16076 non-null  str           
 11  congestion_exit_rating   16076 non-null  str           
 12  time_segment_id          16076 non-null  in

In [4]:
Train = Train.sort_values(by=['view_label', 'datetimestamp_start']).reset_index(drop=True)

print("First few rows of the sorted DataFrame:")
display(Train.head())

First few rows of the sorted DataFrame:


,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train


### Grouping data into continous times before breakage(1 second alowance)

In [5]:
continuous_segments = []
cluster_id_counter = 0

for view_label, group in Train.groupby('view_label'):
    current_cluster = []
    for i, row in group.iterrows():
        if not current_cluster: # First segment in a potential new cluster
            current_cluster.append(row)
        else:
            last_segment_end = current_cluster[-1]['datetimestamp_end']
            current_segment_start = row['datetimestamp_start']

            # --- CORRECTION APPLIED HERE ---
            # Define strict continuity: current segment's start time should be within 2 seconds of the previous segment's end time
            # This accounts for the observed 1-second gap in truly continuous data and flags larger gaps.
            if (current_segment_start - last_segment_end).total_seconds() <= 1:
                current_cluster.append(row)
            else:
                # Continuity broken, finalize the current cluster and start a new one
                if current_cluster:
                    cluster_id_counter += 1
                    start_time = current_cluster[0]['datetimestamp_start']
                    end_time = current_cluster[-1]['datetimestamp_end']
                    duration = end_time - start_time
                    continuous_segments.append({
                        'view_label': view_label,
                        'cluster_id': cluster_id_counter,
                        'start_time': start_time,
                        'end_time': end_time,
                        'total_duration': duration,
                        'num_segments': len(current_cluster)
                    })
                current_cluster = [row] # Start new cluster with the current segment

    # After the loop, finalize any remaining current_cluster
    if current_cluster:
        cluster_id_counter += 1
        start_time = current_cluster[0]['datetimestamp_start']
        end_time = current_cluster[-1]['datetimestamp_end']
        duration = end_time - start_time
        continuous_segments.append({
            'view_label': view_label,
            'cluster_id': cluster_id_counter,
            'start_time': start_time,
            'end_time': end_time,
            'total_duration': duration,
            'num_segments': len(current_cluster)
        })

continuous_segments_df = pd.DataFrame(continuous_segments)

print("Continuous time segments DataFrame created successfully with stricter continuity rules.")
print("First few rows of continuous_segments_df:")
display(continuous_segments_df.head())


Continuous time segments DataFrame created successfully with stricter continuity rules.
First few rows of continuous_segments_df:


,view_label,cluster_id,start_time,end_time,total_duration,num_segments
0,Norman Niles #1,1,2025-10-20 06:00:45,2025-10-20 06:03:00,0 days 00:02:15,3
1,Norman Niles #1,2,2025-10-20 06:03:45,2025-10-20 06:04:59,0 days 00:01:14,2
2,Norman Niles #1,3,2025-10-20 06:05:45,2025-10-20 06:44:59,0 days 00:39:14,40
3,Norman Niles #1,4,2025-10-20 06:45:45,2025-10-20 07:12:00,0 days 00:26:15,27
4,Norman Niles #1,5,2025-10-20 07:12:45,2025-10-20 07:44:59,0 days 00:32:14,33


### Checking the distribution of the segments

In [6]:
norman_niles_segments = continuous_segments_df[continuous_segments_df['view_label'].str.contains('Norman Niles')].copy()

print("Continuous time segments for 'Norman Niles' views:")
display(norman_niles_segments)

print("\nSummary of continuous time segments and breakages for 'Norman Niles' views:")
for view_label, group in norman_niles_segments.groupby('view_label'):
    print(f"\n--- {view_label} ---")
    print(f"Total continuous segments: {len(group)}")
    print(f"Longest continuous segment: {group['total_duration'].max()}")
    print(f"Shortest continuous segment: {group['total_duration'].min()}")
    print(f"Average continuous segment duration: {group['total_duration'].mean()}")

    # Calculate breakages (time gaps between consecutive segments in the same view)
    breakages = []
    sorted_group = group.sort_values(by='start_time').reset_index(drop=True)
    for i in range(1, len(sorted_group)):
        previous_end = sorted_group.loc[i-1, 'end_time']
        current_start = sorted_group.loc[i, 'start_time']
        gap = current_start - previous_end
        if gap.total_seconds() > 1:
            breakages.append(gap)

    if breakages:
        print(f"Number of significant breakages (gaps > 1sec): {len(breakages)}")
        print(f"Total breakage time: {sum(breakages, pd.Timedelta(0))}")
        print(f"Average breakage duration: {sum(breakages, pd.Timedelta(0)) / len(breakages)}")
    else:
        print("No significant breakages detected (all gaps <= 1sec).")

Continuous time segments for 'Norman Niles' views:


,view_label,cluster_id,start_time,end_time,total_duration,num_segments
0,Norman Niles #1,1,2025-10-20 06:00:45,2025-10-20 06:03:00,0 days 00:02:15,3
1,Norman Niles #1,2,2025-10-20 06:03:45,2025-10-20 06:04:59,0 days 00:01:14,2
2,Norman Niles #1,3,2025-10-20 06:05:45,2025-10-20 06:44:59,0 days 00:39:14,40
3,Norman Niles #1,4,2025-10-20 06:45:45,2025-10-20 07:12:00,0 days 00:26:15,27
4,Norman Niles #1,5,2025-10-20 07:12:45,2025-10-20 07:44:59,0 days 00:32:14,33
...,...,...,...,...,...,...
1192,Norman Niles #4,1193,2025-10-26 17:32:42,2025-10-26 17:45:42,0 days 00:13:00,13
1193,Norman Niles #4,1194,2025-10-26 17:46:39,2025-10-26 17:47:38,0 days 00:00:59,1
1194,Norman Niles #4,1195,2025-10-26 17:47:40,2025-10-26 17:52:40,0 days 00:05:00,5
1195,Norman Niles #4,1196,2025-10-26 17:52:42,2025-10-26 17:58:00,0 days 00:05:18,6



Summary of continuous time segments and breakages for 'Norman Niles' views:

--- Norman Niles #1 ---
Total continuous segments: 516
Longest continuous segment: 0 days 00:57:59
Shortest continuous segment: -1 days +23:55:15
Average continuous segment duration: 0 days 00:07:16.784883
Number of significant breakages (gaps > 1sec): 515
Total breakage time: 3 days 21:23:52
Average breakage duration: 0 days 00:10:52.877669902

--- Norman Niles #2 ---
Total continuous segments: 194
Longest continuous segment: 0 days 01:31:00
Shortest continuous segment: 0 days 00:00:13
Average continuous segment duration: 0 days 00:20:17.963917
Number of significant breakages (gaps > 1sec): 193
Total breakage time: 3 days 18:22:09
Average breakage duration: 0 days 00:28:05.642487046

--- Norman Niles #3 ---
Total continuous segments: 207
Longest continuous segment: 0 days 01:55:00
Shortest continuous segment: 0 days 00:00:16
Average continuous segment duration: 0 days 00:19:02.367149
Number of significant br

### Removing rows with less than 22 segments

In [7]:
print("\nNumber of continuous segments 22 minutes and above for 'Norman Niles' views:")
for view_label, group in norman_niles_segments.groupby('view_label'):
    # Filter segments longer than or equal to 22 minutes
    long_segments = group[group['total_duration'] >= pd.Timedelta(minutes=22)]
    print(f"--- {view_label} ---")
    print(f"Segments >= 22 minutes: {len(long_segments)}")
    if not long_segments.empty:
        display(long_segments)



Number of continuous segments 22 minutes and above for 'Norman Niles' views:
--- Norman Niles #1 ---
Segments >= 22 minutes: 35


,view_label,cluster_id,start_time,end_time,total_duration,num_segments
2,Norman Niles #1,3,2025-10-20 06:05:45,2025-10-20 06:44:59,0 days 00:39:14,40
3,Norman Niles #1,4,2025-10-20 06:45:45,2025-10-20 07:12:00,0 days 00:26:15,27
4,Norman Niles #1,5,2025-10-20 07:12:45,2025-10-20 07:44:59,0 days 00:32:14,33
25,Norman Niles #1,26,2025-10-20 11:08:45,2025-10-20 11:46:44,0 days 00:37:59,38
37,Norman Niles #1,38,2025-10-20 13:08:44,2025-10-20 13:33:43,0 days 00:24:59,25
45,Norman Niles #1,46,2025-10-20 15:10:44,2025-10-20 15:42:00,0 days 00:31:16,32
54,Norman Niles #1,55,2025-10-20 17:23:53,2025-10-20 17:47:59,0 days 00:24:06,25
59,Norman Niles #1,60,2025-10-21 06:00:45,2025-10-21 06:32:44,0 days 00:31:59,32
62,Norman Niles #1,63,2025-10-21 07:15:45,2025-10-21 07:41:44,0 days 00:25:59,26
86,Norman Niles #1,87,2025-10-21 11:01:44,2025-10-21 11:39:43,0 days 00:37:59,38


--- Norman Niles #2 ---
Segments >= 22 minutes: 66


,view_label,cluster_id,start_time,end_time,total_duration,num_segments
517,Norman Niles #2,518,2025-10-20 06:03:45,2025-10-20 06:45:00,0 days 00:41:15,42
518,Norman Niles #2,519,2025-10-20 06:45:45,2025-10-20 07:52:45,0 days 01:07:00,67
520,Norman Niles #2,521,2025-10-20 08:23:46,2025-10-20 08:55:00,0 days 00:31:14,32
522,Norman Niles #2,523,2025-10-20 09:23:46,2025-10-20 09:51:00,0 days 00:27:14,28
523,Norman Niles #2,524,2025-10-20 09:51:46,2025-10-20 10:15:59,0 days 00:24:13,24
...,...,...,...,...,...,...
670,Norman Niles #2,671,2025-10-26 07:38:46,2025-10-26 08:21:00,0 days 00:42:14,43
679,Norman Niles #2,680,2025-10-26 10:25:46,2025-10-26 10:59:17,0 days 00:33:31,34
681,Norman Niles #2,682,2025-10-26 11:01:46,2025-10-26 11:37:45,0 days 00:35:59,36
699,Norman Niles #2,700,2025-10-26 14:27:47,2025-10-26 15:11:43,0 days 00:43:56,44


--- Norman Niles #3 ---
Segments >= 22 minutes: 57


,view_label,cluster_id,start_time,end_time,total_duration,num_segments
714,Norman Niles #3,715,2025-10-20 06:15:45,2025-10-20 07:52:43,0 days 01:36:58,97
716,Norman Niles #3,717,2025-10-20 08:23:44,2025-10-20 08:55:00,0 days 00:31:16,32
718,Norman Niles #3,719,2025-10-20 09:23:43,2025-10-20 09:51:00,0 days 00:27:17,28
719,Norman Niles #3,720,2025-10-20 09:51:43,2025-10-20 10:15:42,0 days 00:23:59,24
723,Norman Niles #3,724,2025-10-20 11:01:42,2025-10-20 11:46:43,0 days 00:45:01,45
724,Norman Niles #3,725,2025-10-20 12:09:43,2025-10-20 12:45:00,0 days 00:35:17,36
727,Norman Niles #3,728,2025-10-20 13:05:42,2025-10-20 13:33:41,0 days 00:27:59,28
733,Norman Niles #3,734,2025-10-20 15:10:42,2025-10-20 16:40:41,0 days 01:29:59,90
735,Norman Niles #3,736,2025-10-20 17:02:42,2025-10-20 17:51:00,0 days 00:48:18,49
737,Norman Niles #3,738,2025-10-21 06:00:44,2025-10-21 06:32:43,0 days 00:31:59,32


--- Norman Niles #4 ---
Segments >= 22 minutes: 54


,view_label,cluster_id,start_time,end_time,total_duration,num_segments
918,Norman Niles #4,919,2025-10-20 06:03:44,2025-10-20 07:52:43,0 days 01:48:59,109
920,Norman Niles #4,921,2025-10-20 08:23:44,2025-10-20 08:55:00,0 days 00:31:16,32
922,Norman Niles #4,923,2025-10-20 09:23:44,2025-10-20 09:51:00,0 days 00:27:16,28
928,Norman Niles #4,929,2025-10-20 11:01:44,2025-10-20 11:46:44,0 days 00:45:00,45
929,Norman Niles #4,930,2025-10-20 12:09:44,2025-10-20 12:45:00,0 days 00:35:16,36
932,Norman Niles #4,933,2025-10-20 13:05:44,2025-10-20 13:33:43,0 days 00:27:59,28
933,Norman Niles #4,934,2025-10-20 13:55:44,2025-10-20 14:21:00,0 days 00:25:16,26
938,Norman Niles #4,939,2025-10-20 15:10:44,2025-10-20 16:04:42,0 days 00:53:58,54
939,Norman Niles #4,940,2025-10-20 16:04:44,2025-10-20 16:41:43,0 days 00:36:59,37
950,Norman Niles #4,951,2025-10-21 06:00:44,2025-10-21 06:32:43,0 days 00:31:59,32


### View now the new data that we will be using 

In [8]:
import pandas as pd

cleaned_train_list = []

# Filter norman_niles_segments for those >= 22 minutes
long_norman_niles_clusters = norman_niles_segments[norman_niles_segments['total_duration'] >= pd.Timedelta(minutes=22)]

print(f"Found {len(long_norman_niles_clusters)} continuous segments of 22 minutes or longer in 'Norman Niles' views.")

# Iterate through each long continuous segment identified
for index, cluster_row in long_norman_niles_clusters.iterrows():
    view_label = cluster_row['view_label']
    start_time = cluster_row['start_time']
    end_time = cluster_row['end_time']

    # Filter the original Train DataFrame for rows within this cluster's time range and view_label
    cluster_segments = Train[
        (Train['view_label'] == view_label) &
        (Train['datetimestamp_start'] >= start_time) &
        (Train['datetimestamp_end'] <= end_time)
    ]

    # Append these segments to our list
    if not cluster_segments.empty:
        cleaned_train_list.append(cluster_segments)

# Concatenate all collected segments into the final cleaned_train DataFrame
if cleaned_train_list:
    cleaned_train = pd.concat(cleaned_train_list, ignore_index=True)
    print(f"Created 'cleaned_train' DataFrame with {len(cleaned_train)} rows.")
    print("First 5 rows of the cleaned_train DataFrame:")
    display(cleaned_train.head())
    print("Last 5 rows of the cleaned_train DataFrame:")
    display(cleaned_train.tail())
    print("Info of the cleaned_train DataFrame:")
    cleaned_train.info()
else:
    print("No segments of 22 minutes or longer found. 'cleaned_train' DataFrame is empty.")


Found 212 continuous segments of 22 minutes or longer in 'Norman Niles' views.
Created 'cleaned_train' DataFrame with 9014 rows.
First 5 rows of the cleaned_train DataFrame:


,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,SIsHaebOK7mfnvgPoaTSqQ0,Norman Niles #1,time_segment_5_Norman Niles #1_congestion_ente...,time_segment_5_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-05-45.mp4,2025-10-20 06:05:45,2025-10-20 06:05:45,2025-10-20 06:06:44,2025-10-20,none,free flowing,free flowing,5,train
1,EI0HadH5KqHXnvgP76fxyQI,Norman Niles #1,time_segment_6_Norman Niles #1_congestion_ente...,time_segment_6_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-06-45.mp4,2025-10-20 06:06:45,2025-10-20 06:06:45,2025-10-20 06:07:44,2025-10-20,low,free flowing,free flowing,6,train
2,A40HaYaBG8jtnvgP8NiHuAw,Norman Niles #1,time_segment_7_Norman Niles #1_congestion_ente...,time_segment_7_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-07-45.mp4,2025-10-20 06:07:45,2025-10-20 06:07:45,2025-10-20 06:08:44,2025-10-20,none,free flowing,free flowing,7,train
3,04kHacmOLrXOnvgP3YW5kQs,Norman Niles #1,time_segment_8_Norman Niles #1_congestion_ente...,time_segment_8_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-08-45.mp4,2025-10-20 06:08:45,2025-10-20 06:08:45,2025-10-20 06:09:44,2025-10-20,none,free flowing,free flowing,8,train
4,EI0HaeWIIb-envgP6uGTmAE,Norman Niles #1,time_segment_9_Norman Niles #1_congestion_ente...,time_segment_9_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-09-45.mp4,2025-10-20 06:09:45,2025-10-20 06:09:45,2025-10-20 06:10:44,2025-10-20,none,free flowing,free flowing,9,train


Last 5 rows of the cleaned_train DataFrame:


,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
9009,EI0HaZ-FJqHXnvgP76fxyQI,Norman Niles #4,time_segment_4866_Norman Niles #4_congestion_e...,time_segment_4866_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-15-56-45.mp4,2025-10-26 15:56:45,2025-10-26 15:56:44,2025-10-26 15:57:44,2025-10-26,low,light delay,free flowing,4866,train
9010,8YkHaYLVA7-envgP6uGTmAE,Norman Niles #4,time_segment_4867_Norman Niles #4_congestion_e...,time_segment_4867_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-15-57-45.mp4,2025-10-26 15:57:45,2025-10-26 15:57:44,2025-10-26 15:58:44,2025-10-26,medium,light delay,free flowing,4867,train
9011,_YkHacT-COK2nvgPsZGZ4Aw,Norman Niles #4,time_segment_4868_Norman Niles #4_congestion_e...,time_segment_4868_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-15-58-45.mp4,2025-10-26 15:58:45,2025-10-26 15:58:44,2025-10-26 15:59:44,2025-10-26,none,free flowing,free flowing,4868,train
9012,q4sHaduLA_K1ld8PufLf6AI,Norman Niles #4,time_segment_4869_Norman Niles #4_congestion_e...,time_segment_4869_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-15-59-45.mp4,2025-10-26 15:59:45,2025-10-26 15:59:44,2025-10-26 16:00:59,2025-10-26,none,free flowing,free flowing,4869,train
9013,_okHae7PGvK1ld8PufLf6AI,Norman Niles #4,time_segment_4870_Norman Niles #4_congestion_e...,time_segment_4870_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-16-00-45.mp4,2025-10-26 16:00:45,2025-10-26 16:00:44,2025-10-26 16:01:00,2025-10-26,none,free flowing,free flowing,4870,train


Info of the cleaned_train DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 9014 entries, 0 to 9013
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   responseId               9014 non-null   str           
 1   view_label               9014 non-null   str           
 2   ID_enter                 9014 non-null   str           
 3   ID_exit                  9014 non-null   str           
 4   videos                   9014 non-null   str           
 5   video_time               9014 non-null   str           
 6   datetimestamp_start      9014 non-null   datetime64[us]
 7   datetimestamp_end        9014 non-null   datetime64[us]
 8   date                     9014 non-null   str           
 9   signaling                9014 non-null   str           
 10  congestion_enter_rating  9014 non-null   str           
 11  congestion_exit_rating   9014 non-null   str           
 12  time_seg

### Save to csv

In [9]:
cleaned_train.to_csv("cleaned_train.csv", index=False)

In [10]:
display(cleaned_train.congestion_enter_rating.value_counts())
display(cleaned_train.congestion_exit_rating.value_counts())

congestion_enter_rating
free flowing      5246
moderate delay    1431
heavy delay       1184
light delay       1153
Name: count, dtype: int64

congestion_exit_rating
free flowing      8548
moderate delay     188
light delay        150
heavy delay        128
Name: count, dtype: int64

# APPROXIMATELY 30 SECONDS TO RUN EVERYTHING